In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os

In [2]:
df = pd.read_parquet('../data/processed/merged.parquet')

# Create a combined text field for embedding
df['combined_text'] = (
    df['product_title'].fillna('') + ' ' +
    df['text'].fillna('')
).str.strip()

documents = df['combined_text'].tolist()

In [8]:
with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

In [9]:
with open("documents.pkl", "rb") as f:
    documents = pickle.load(f)

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')
# print("Generating embeddings...")
# embeddings = model.encode(documents, show_progress_bar=True, batch_size=256)
# np.save("embeddings.npy", embeddings)
# print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# pd.DataFrame(embeddings).head()
embeddings = np.load("../data/processed/embeddings.npy")

In [7]:
dimension = embeddings.shape[1]  # 384 for all-MiniLM-L6-v2

index = faiss.IndexFlatL2(dimension)   # L2 (Euclidean) distance
# or use cosine similarity:
# index = faiss.IndexFlatIP(dimension) # Inner product (cosine if normalized)

index.add(embeddings)
print(f"Total vectors indexed: {index.ntotal}")

Total vectors indexed: 701528


In [13]:
os.makedirs('../data/processed/faiss_index', exist_ok=True)

faiss.write_index(index, '../data/processed/faiss_index/index.faiss')

# Save the dataframe so you can retrieve metadata after search
df.to_parquet('../data/processed/faiss_index/metadata.parquet', index=False)

print("Index and metadata saved.")

Index and metadata saved.


In [5]:
def semantic_search(query, top_k=5):
    query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    
    results = df.iloc[indices[0]].copy()
    results['distance'] = distances[0]
    
    return pd.DataFrame(results)

In [10]:
results = semantic_search("Sunscreen", top_k=5)
results

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details,combined_text,distance
365958,5.0,Must-Have for Humid Summers!,This sun stick is AMAZING. It smells so good; ...,False,JM solution Marine Luminous Pearl Sun Stick SP...,4.2,None,[],JM Solution,"{""Product Benefits"": ""Whitening"", ""Item Weight...",JM solution Marine Luminous Pearl Sun Stick SP...,0.515759
474161,5.0,I came back to buy another one. ❤,Great product works just like it should. Fill...,True,"Large Essential Oil Diffuser 1800 ML, 14 Color...",4.0,None,[],Aromaliferow,"{""Brand"": ""Aromaliferow"", ""Special Feature"": ""...","Large Essential Oil Diffuser 1800 ML, 14 Color...",0.521623
257650,5.0,The kids loved these.,It was for my niece birthday party.,True,CHuangQi 20 pcs Mouse Ears Headband Solid Blac...,4.7,18.99,[],CHuangQi,"{""Color"": ""Pink & Black"", ""Age Range (Descript...",CHuangQi 20 pcs Mouse Ears Headband Solid Blac...,0.524359
695997,5.0,Very cute,"Very nice, they look amazing. And are very str...",True,Christmas Almond Press on Nails Long Stiletto ...,3.6,None,[],HJKDSFD,"{""Size"": ""Large"", ""Material"": ""Acrylic"", ""Bran...",Christmas Almond Press on Nails Long Stiletto ...,0.531206
131271,5.0,Five Stars,Fine product,True,Philips Norelco arcitec 1090 Men's Shaving System,4.0,None,"[From the Manufacturer, Select the right shavi...",PHILIPS,"{""Brand"": ""PHILIPS"", ""Power Source"": ""Battery ...",Philips Norelco arcitec 1090 Men's Shaving Sys...,0.534254
